# 02 — Feature Store Validation
## Air Quality & Public Health Impact Pipeline
### France · Spain · Belgium · Netherlands · Germany · 2019–2023

---

### Purpose of this notebook

This notebook sits between the Spark pipeline and model training.  
Its job is to **verify that the feature store produced by `spark_join_features.py` is correct** before any model is trained on it.

This is a critical MLOps checkpoint. Training a model on a corrupted feature store — one with data leakage, wrong lag directions, or schema mismatches — produces results that look fine but are fundamentally invalid. Catching problems here is far cheaper than debugging a deployed model.

Specifically, this notebook verifies:

1. **Schema** — all expected columns are present with the right dtypes.
2. **No data leakage** — lag and rolling features look strictly backwards; the target looks strictly forwards.
3. **Target correctness** — `pm25_exceedance_t1` and `t2` match the raw `pm25_mean` values they were derived from.
4. **Missing value patterns** — null rates per column; first-row nulls from lags are expected and acceptable.
5. **Feature distributions** — lag and rolling features have the right statistical relationship to their source column.
6. **No duplicate rows** — each (station_id, date) pair should appear exactly once.
7. **Temporal continuity** — no unexpected gaps in the time series per station.
8. **Baseline model experiment** — a minimal LightGBM run on the feature store to confirm the data is learnable before committing to full training in `train.py`.

---

### Relationship to other notebooks and scripts

```
spark_join_features.py  →  [THIS NOTEBOOK]  →  train.py
                                              →  did_analysis.py
                                              →  iv_analysis.py
```

The feature store must pass all checks in this notebook before `train.py` is run.  
If any check fails, fix the root cause in `spark_join_features.py` and re-run the Spark job.

---

### Notebook structure

| Section | What it checks |
|---|---|
| 0. Setup | Imports, paths, expected schema definition |
| 1. Load | Load feature store, quick size check |
| 2. Schema | Column presence, dtypes, no unexpected extras |
| 3. Duplicates | One row per (station_id, date) |
| 4. Missing Values | Null rates per column; first-row lag nulls are expected |
| 5. Leakage Check | Lag features look backward; rolling windows exclude today; targets look forward |
| 6. Target Correctness | Exceedance flags match raw PM2.5 values |
| 7. Feature Distributions | Lag/rolling features have correct statistical relationship to source |
| 8. Temporal Continuity | No unexpected gaps in station time series |
| 9. Baseline Model | Quick LightGBM run to verify data is learnable |
| 10. Validation Summary | Pass/fail report for all checks |

---
## 0. Setup

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 4)

from src.utils.paths import PROCESSED_FEATURES

print("Setup complete.")
print(f"Feature store path: {PROCESSED_FEATURES}")

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — mirrors the constants in spark_join_features.py
# If you change these in the Spark script, change them here too.
# ---------------------------------------------------------------------------

COUNTRIES      = ["FR", "ES", "BE", "NL", "DE"]
START_DATE     = "2019-01-01"
END_DATE       = "2023-12-31"
PM25_THRESHOLD = 25.0  # µg/m³ — EU daily limit, used to compute exceedance targets

# Lag days and rolling windows — must match spark_join_features.py
LAG_DAYS     = [1, 2, 7]
ROLLING_DAYS = [7, 14]

# Source columns for which lags and rolling features were computed
FEATURE_POLLUTANTS = ["pm25_mean", "pm10_mean", "no2_mean", "o3_mean", "so2_mean"]
FEATURE_METEO      = ["temp_2m", "wind_speed", "surface_pressure",
                      "precipitation", "boundary_layer_height", "low_blh_flag"]

# ---------------------------------------------------------------------------
# Expected schema — every key in this dict must be present in the feature store
# Values are acceptable pandas dtypes (or 'any' for flexible checks)
# ---------------------------------------------------------------------------
EXPECTED_COLUMNS = {
    # Identifiers
    "station_id":    "object",
    "country_code":  "object",
    "date":          "datetime64[ns]",
    # Calendar
    "year":          "int",
    "month":         "int",
    "day_of_week":   "int",
    "season":        "object",
    "is_weekend":    "int",
    "heating_season":"int",
    # Core pollutants
    "pm25_mean":     "float",
    "pm10_mean":     "float",
    "no2_mean":      "float",
    "o3_mean":       "float",
    "so2_mean":      "float",
    # ERA5 meteorology
    "temp_2m":                   "float",
    "wind_speed":                "float",
    "surface_pressure":          "float",
    "precipitation":             "float",
    "boundary_layer_height":     "float",
    "low_blh_flag":              "int",
    # Targets
    "pm25_exceedance_t1": "int",
    "pm25_exceedance_t2": "int",
    # Health join
    "weekly_deaths": "float",  # nullable — left join
    "nuts3_code":    "object",
    "week_of_year":  "int",
}

# ---------------------------------------------------------------------------
# Validation results collector
# Each check appends to this list; Section 10 prints the full report.
# ---------------------------------------------------------------------------
VALIDATION_RESULTS = []  # list of {check, status, detail}

def record(check: str, passed: bool, detail: str = ""):
    status = "PASS" if passed else "FAIL"
    VALIDATION_RESULTS.append({"check": check, "status": status, "detail": detail})
    icon = "✓" if passed else "✗"
    print(f"  {icon} [{status}] {check}" + (f" — {detail}" if detail else ""))

print("Configuration set. Validation collector ready.")

---
## 1. Load Feature Store

In [ ]:
# ---------------------------------------------------------------------------
# Load the full feature store for our five countries.
# Partition pruning on country_code avoids reading irrelevant partitions.
# ---------------------------------------------------------------------------
print("Loading feature store...")

df = pd.read_parquet(
    PROCESSED_FEATURES,
    filters=[("country_code", "in", COUNTRIES)],
    engine="pyarrow",
)

df["date"] = pd.to_datetime(df["date"])
df = df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)].copy()
df = df.sort_values(["station_id", "date"]).reset_index(drop=True)

print(f"Rows      : {len(df):,}")
print(f"Columns   : {len(df.columns)}")
print(f"Stations  : {df['station_id'].nunique():,}")
print(f"Countries : {sorted(df['country_code'].unique())}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")

# Basic size sanity check
# 1,000 stations × 365 days × 5 years = 1.8M rows minimum expected
size_ok = len(df) > 500_000
record("Feature store has sufficient rows", size_ok,
       f"{len(df):,} rows (expected > 500,000)")

---
## 2. Schema Validation

Verify that every expected column is present and has an acceptable dtype.  
Schema mismatches between the Spark output and what `train.py` expects are a common silent failure mode.

In [ ]:
# ---------------------------------------------------------------------------
# 2.1 Check all expected columns are present
# ---------------------------------------------------------------------------
print("\n--- 2.1 Column presence ---")

missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
record("All expected columns present",
       len(missing_cols) == 0,
       f"Missing: {missing_cols}" if missing_cols else "All present")

if missing_cols:
    print(f"\n  Missing columns: {missing_cols}")
    print("  → Check spark_join_features.py for the step that produces each missing column.")

In [ ]:
# ---------------------------------------------------------------------------
# 2.2 Check all lag columns are present
#
# Expected pattern: {source_col}_lag{n} for every source column and lag day.
# E.g. pm25_mean_lag1, pm25_mean_lag2, pm25_mean_lag7
# ---------------------------------------------------------------------------
print("\n--- 2.2 Lag column presence ---")

all_source_cols = FEATURE_POLLUTANTS + FEATURE_METEO
expected_lag_cols = [f"{col}_lag{n}" for col in all_source_cols for n in LAG_DAYS
                     if col in df.columns]  # only check if source col exists
missing_lag_cols  = [c for c in expected_lag_cols if c not in df.columns]

record("All lag columns present",
       len(missing_lag_cols) == 0,
       f"{len(expected_lag_cols) - len(missing_lag_cols)}/{len(expected_lag_cols)} present"
       + (f". Missing: {missing_lag_cols[:5]}..." if missing_lag_cols else ""))

In [ ]:
# ---------------------------------------------------------------------------
# 2.3 Check rolling window columns are present
#
# Expected pattern: {source_col}_rolling{n}d
# Rolling features are only computed for pollutants + BLH + temperature
# (see spark_join_features.py — rolling_cols definition)
# ---------------------------------------------------------------------------
print("\n--- 2.3 Rolling column presence ---")

rolling_source = FEATURE_POLLUTANTS + ["boundary_layer_height", "temp_2m"]
expected_rolling_cols = [f"{col}_rolling{n}d"
                         for col in rolling_source
                         for n in ROLLING_DAYS
                         if col in df.columns]
missing_rolling_cols  = [c for c in expected_rolling_cols if c not in df.columns]

record("All rolling columns present",
       len(missing_rolling_cols) == 0,
       f"{len(expected_rolling_cols) - len(missing_rolling_cols)}/{len(expected_rolling_cols)} present"
       + (f". Missing: {missing_rolling_cols[:5]}..." if missing_rolling_cols else ""))

In [ ]:
# ---------------------------------------------------------------------------
# 2.4 Dtype checks for key columns
#
# We check broad type families (numeric, string, datetime) rather than
# exact dtypes, since Parquet and pandas may use float32 vs float64, etc.
# ---------------------------------------------------------------------------
print("\n--- 2.4 Column dtypes ---")

dtype_issues = []
for col, expected_type in EXPECTED_COLUMNS.items():
    if col not in df.columns:
        continue
    actual = df[col].dtype
    # Broad family checks
    if expected_type == "float" and not pd.api.types.is_float_dtype(actual):
        dtype_issues.append(f"{col}: expected float, got {actual}")
    elif expected_type == "int" and not pd.api.types.is_integer_dtype(actual):
        # integer cols may be float if nulls exist — acceptable for nullable ints
        if not pd.api.types.is_float_dtype(actual):
            dtype_issues.append(f"{col}: expected int/float, got {actual}")
    elif expected_type == "object" and actual != object:
        dtype_issues.append(f"{col}: expected string/object, got {actual}")
    elif expected_type == "datetime64[ns]" and not pd.api.types.is_datetime64_any_dtype(actual):
        dtype_issues.append(f"{col}: expected datetime, got {actual}")

record("Column dtypes are correct",
       len(dtype_issues) == 0,
       "; ".join(dtype_issues) if dtype_issues else "All dtypes OK")

if dtype_issues:
    for issue in dtype_issues:
        print(f"  → {issue}")

---
## 3. Duplicate Row Check

Each (station_id, date) pair must appear exactly once.  
Duplicates indicate a bug in the Spark join logic — typically a many-to-many join that produced cartesian product rows.

In [ ]:
print("\n--- 3. Duplicate rows ---")

n_rows    = len(df)
n_unique  = df[["station_id", "date"]].drop_duplicates().shape[0]
n_dupes   = n_rows - n_unique

record("No duplicate (station_id, date) rows",
       n_dupes == 0,
       f"{n_dupes:,} duplicates found" if n_dupes > 0 else "No duplicates")

if n_dupes > 0:
    # Show the worst offenders to help diagnose
    dupes = (
        df.groupby(["station_id", "date"])
        .size()
        .reset_index(name="count")
        .query("count > 1")
        .sort_values("count", ascending=False)
    )
    print(f"  Top duplicate rows:")
    print(dupes.head(10).to_string(index=False))
    print("  → Check spark_join_features.py join logic — likely a health join producing duplicates.")

---
## 4. Missing Value Analysis

Not all nulls are problems — some are expected:
- **Lag columns**: the first `n` rows of each station's time series will be null (there is no t-1 for the first day). This is correct.
- **Rolling columns**: similarly, the first `n-1` rows will be null for an n-day rolling window.
- **`weekly_deaths`**: null where the NUTS3-week health data is missing — expected for some regions.

What is **not** acceptable:
- High null rates in `pm25_mean` (> 30%) — suggests the EEA filter was too aggressive.
- Any nulls in `station_id`, `date`, `country_code` — these are primary keys.
- Any nulls in target columns `pm25_exceedance_t1/t2` — these should have been dropped in `add_target_variables()`.

In [ ]:
# ---------------------------------------------------------------------------
# 4.1 Null rates across all columns
# ---------------------------------------------------------------------------
print("\n--- 4.1 Null rates ---")

null_rates = (
    df.isnull().mean() * 100
).round(2).sort_values(ascending=False)

# Display columns with any nulls
cols_with_nulls = null_rates[null_rates > 0]
print(f"Columns with null values: {len(cols_with_nulls)} / {len(df.columns)}")
print(cols_with_nulls.to_string())

In [ ]:
# ---------------------------------------------------------------------------
# 4.2 Critical null checks
# ---------------------------------------------------------------------------
print("\n--- 4.2 Critical null checks ---")

# Primary keys must never be null
for pk_col in ["station_id", "date", "country_code"]:
    if pk_col in df.columns:
        null_count = df[pk_col].isnull().sum()
        record(f"No nulls in primary key '{pk_col}'",
               null_count == 0,
               f"{null_count:,} nulls found" if null_count > 0 else "OK")

# Target columns must be null-free (spark_join_features.py drops null-target rows)
for target in ["pm25_exceedance_t1", "pm25_exceedance_t2"]:
    if target in df.columns:
        null_count = df[target].isnull().sum()
        record(f"No nulls in target '{target}'",
               null_count == 0,
               f"{null_count:,} nulls — check add_target_variables() in spark_join_features.py"
               if null_count > 0 else "OK")

# pm25_mean null rate should be below 30%
if "pm25_mean" in df.columns:
    pm25_null_rate = df["pm25_mean"].isnull().mean() * 100
    record("pm25_mean null rate < 30%",
           pm25_null_rate < 30,
           f"{pm25_null_rate:.1f}% null — investigate EEA data quality filter")

# ERA5 meteo join success rate
if "temp_2m" in df.columns:
    era5_null_rate = df["temp_2m"].isnull().mean() * 100
    record("ERA5 join success rate > 90%",
           era5_null_rate < 10,
           f"{100 - era5_null_rate:.1f}% of rows have ERA5 data")

In [ ]:
# ---------------------------------------------------------------------------
# 4.3 Visualise null pattern — heatmap of null rates by column group
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 5))

# Sort columns into groups for readability
col_order = (
    [c for c in df.columns if c in ["station_id","date","country_code","year","month"]]
    + [c for c in df.columns if "pm25" in c and "lag" not in c and "rolling" not in c]
    + [c for c in df.columns if any(m in c for m in ["temp","wind","blh","boundary","precip"]) and "lag" not in c and "rolling" not in c]
    + [c for c in df.columns if "lag1" in c][:8]  # sample of lag cols
    + [c for c in df.columns if "rolling7" in c][:5]  # sample of rolling cols
    + [c for c in df.columns if "exceedance" in c or "deaths" in c]
)
col_order = [c for c in col_order if c in df.columns]  # safety filter

null_subset = (df[col_order].isnull().mean() * 100).round(1)
null_matrix = null_subset.values.reshape(1, -1)

sns.heatmap(
    null_matrix,
    ax=ax,
    xticklabels=col_order,
    yticklabels=["Null %"],
    annot=True, fmt=".0f",
    cmap="Reds",
    vmin=0, vmax=100,
    linewidths=0.5,
    cbar_kws={"label": "% null"},
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_title("Null rate (%) by column — lag nulls at series start are expected")
plt.tight_layout()
plt.show()

---
## 5. Data Leakage Check

This is the most important validation in the notebook.

**Data leakage** means that a feature at time `t` contains information from time `t+1` or later.  
A model trained on leaky features will appear to perform extremely well during validation but will fail completely in production — it learned to "predict" the future using future information it won't have at inference time.

Three potential leakage sources in this feature store:
1. **Lag features** — must look strictly backward (t-1, t-2, t-7). If `pm25_mean_lag1` at row t equals `pm25_mean` at row t, the lag direction is wrong.
2. **Rolling features** — `rowsBetween(-(n-1), -1)` excludes today. If a rolling feature equals the same-day value, today was incorrectly included.
3. **Target** — `pm25_exceedance_t1` is the exceedance flag for **tomorrow** (t+1). It must not be included as a feature in `train.py`.

In [ ]:
# ---------------------------------------------------------------------------
# 5.1 Lag feature direction check
#
# For each station, lag(pm25_mean, 1) at row t should equal pm25_mean at row t-1.
# We verify this on a sample of stations.
#
# Method: pick one station, sort by date, then check that
#   df["pm25_mean_lag1"].shift(-1) == df["pm25_mean"]
# (shifting lag1 forward by one should recover the original series)
# ---------------------------------------------------------------------------
print("\n--- 5.1 Lag feature direction ---")

if "pm25_mean_lag1" in df.columns:
    # Test on 10 random stations
    test_stations = df["station_id"].drop_duplicates().sample(n=min(10, df["station_id"].nunique()), random_state=42)
    lag_errors = 0

    for station in test_stations:
        sub = df[df["station_id"] == station].sort_values("date").reset_index(drop=True)
        # lag1 at position i should equal pm25_mean at position i-1
        # Compare lag1[1:] with pm25_mean[:-1]
        lag1   = sub["pm25_mean_lag1"].iloc[1:].values
        source = sub["pm25_mean"].iloc[:-1].values
        # Allow NaN equality (both NaN counts as match)
        mismatch = np.sum(
            ~(np.isclose(lag1, source, equal_nan=True) |
              (np.isnan(lag1) & np.isnan(source)))
        )
        if mismatch > 0:
            lag_errors += 1

    record("Lag features look strictly backward (t-1)",
           lag_errors == 0,
           f"{lag_errors}/10 test stations have lag direction errors"
           if lag_errors > 0
           else "Verified on 10 random stations")
else:
    record("Lag features look strictly backward (t-1)", False,
           "pm25_mean_lag1 not found — cannot check")

In [ ]:
# ---------------------------------------------------------------------------
# 5.2 Rolling window excludes today
#
# The 7-day rolling mean at time t is the mean of [t-6, t-5, ..., t-1].
# It must NOT include t.
#
# Detection method:
# If a station has a single extreme PM2.5 spike on one day, the rolling mean
# on that same day should NOT reflect the spike (because today is excluded).
# But the rolling mean on the NEXT day SHOULD include yesterday's spike.
#
# We use a simpler proxy: compute the correlation between pm25_mean and
# pm25_mean_rolling7d. If rolling includes today, the correlation will be
# very high (> 0.99). If it excludes today correctly, it will be high
# but not perfect (typically 0.7–0.95).
# ---------------------------------------------------------------------------
print("\n--- 5.2 Rolling window excludes today ---")

if "pm25_mean_rolling7d" in df.columns:
    valid = df[["pm25_mean", "pm25_mean_rolling7d"]].dropna()
    r = valid["pm25_mean"].corr(valid["pm25_mean_rolling7d"])

    # If r > 0.99, same-day value is almost certainly included
    # (rolling mean of 7 values including today would correlate very strongly)
    rolling_ok = r < 0.99
    record("7-day rolling mean excludes today (r < 0.99)",
           rolling_ok,
           f"r(pm25_mean, rolling7d) = {r:.4f}"
           + (" — possible leakage" if not rolling_ok else " — OK"))
else:
    record("7-day rolling mean excludes today", False,
           "pm25_mean_rolling7d not found")

In [ ]:
# ---------------------------------------------------------------------------
# 5.3 Target looks strictly forward
#
# pm25_exceedance_t1 at row t is derived from pm25_mean at row t+1.
# Verify: for each station, the target flag at position i should equal
# (pm25_mean[i+1] > 25) — i.e. tomorrow's value, not today's.
#
# Also verify that pm25_mean itself is NOT in the target column
# (the target was derived by lead(), not from the same row).
# ---------------------------------------------------------------------------
print("\n--- 5.3 Target looks strictly forward ---")

if "pm25_exceedance_t1" in df.columns and "pm25_mean" in df.columns:
    test_stations = df["station_id"].drop_duplicates().sample(
        n=min(10, df["station_id"].nunique()), random_state=99
    )
    target_errors = 0

    for station in test_stations:
        sub = df[df["station_id"] == station].sort_values("date").reset_index(drop=True)
        if len(sub) < 3:
            continue
        # Target at position i should be (pm25_mean[i+1] > threshold)
        expected_target = (sub["pm25_mean"].shift(-1) > PM25_THRESHOLD).astype(float)
        actual_target   = sub["pm25_exceedance_t1"].astype(float)
        # Compare on non-null rows (last row has null target by design)
        mask = sub["pm25_exceedance_t1"].notna() & sub["pm25_mean"].notna()
        if mask.sum() == 0:
            continue
        mismatch = (expected_target[mask] != actual_target[mask]).sum()
        if mismatch > 0:
            target_errors += 1

    record("Target pm25_exceedance_t1 uses t+1 (not t)",
           target_errors == 0,
           f"{target_errors}/10 stations have target direction errors"
           if target_errors > 0
           else "Verified on 10 random stations")
else:
    record("Target pm25_exceedance_t1 uses t+1", False, "Required columns not found")

---
## 6. Target Correctness

Verify the binary exceedance targets match the raw PM2.5 values they were derived from,  
and that t+2 target is correctly one day further ahead than t+1.

In [ ]:
# ---------------------------------------------------------------------------
# 6.1 Target value distribution sanity check
#
# Both targets must be binary (0 or 1 only).
# The t+1 and t+2 exceedance rates should be close but not identical
# (48h forecast is slightly less certain than 24h → slightly different rate).
# ---------------------------------------------------------------------------
print("\n--- 6.1 Target value distribution ---")

for target in ["pm25_exceedance_t1", "pm25_exceedance_t2"]:
    if target not in df.columns:
        continue
    unique_vals = sorted(df[target].dropna().unique())
    is_binary   = set(unique_vals).issubset({0, 1, 0.0, 1.0})
    pos_rate    = df[target].mean() * 100
    record(f"{target} is binary (0/1 only)",
           is_binary,
           f"Unique values: {unique_vals}" if not is_binary
           else f"Positive rate: {pos_rate:.2f}%")

# t+1 and t+2 rates should be similar but not identical
if "pm25_exceedance_t1" in df.columns and "pm25_exceedance_t2" in df.columns:
    rate_t1 = df["pm25_exceedance_t1"].mean() * 100
    rate_t2 = df["pm25_exceedance_t2"].mean() * 100
    diff    = abs(rate_t1 - rate_t2)
    # If rates are identical, t+2 may have been copied from t+1
    record("t+1 and t+2 targets differ (not identical)",
           diff > 0.01,
           f"t+1: {rate_t1:.2f}%, t+2: {rate_t2:.2f}% (diff: {diff:.3f}pp)")

In [ ]:
# ---------------------------------------------------------------------------
# 6.2 Visual check — PM2.5 on danger vs safe days
#
# On rows where pm25_exceedance_t1 = 1, the NEXT day's PM2.5 should be > 25.
# We verify this by plotting the distribution of tomorrow's PM2.5
# split by today's exceedance flag.
# ---------------------------------------------------------------------------
if "pm25_exceedance_t1" in df.columns and "pm25_mean" in df.columns:
    # Reconstruct tomorrow's PM2.5 by shifting within each station
    df_sorted = df.sort_values(["station_id", "date"])
    df_sorted["pm25_tomorrow"] = df_sorted.groupby("station_id")["pm25_mean"].shift(-1)

    sample_check = df_sorted.dropna(subset=["pm25_tomorrow", "pm25_exceedance_t1"]).sample(
        n=min(50_000, len(df_sorted)), random_state=42
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: distribution of tomorrow's PM2.5 by exceedance flag
    for flag, label, color in [(0, "t+1 = 0 (safe)", "steelblue"), (1, "t+1 = 1 (danger)", "tomato")]:
        vals = sample_check.loc[sample_check["pm25_exceedance_t1"] == flag, "pm25_tomorrow"].dropna()
        axes[0].hist(vals, bins=60, alpha=0.6, color=color, label=label, density=True)
    axes[0].axvline(PM25_THRESHOLD, color="red", linestyle="--", linewidth=1.5)
    axes[0].set_xlabel("Tomorrow's PM2.5 (µg/m³)")
    axes[0].set_ylabel("Density")
    axes[0].set_title("Tomorrow's PM2.5 distribution by exceedance flag\n"
                      "(danger flag = 1 should sit right of the threshold)")
    axes[0].legend()

    # Right: scatter plot (today's PM2.5 vs tomorrow's PM2.5, coloured by flag)
    scatter_sample = sample_check.sample(n=min(5000, len(sample_check)), random_state=0)
    colors_map = scatter_sample["pm25_exceedance_t1"].map({0: "steelblue", 1: "tomato"})
    axes[1].scatter(scatter_sample["pm25_mean"], scatter_sample["pm25_tomorrow"],
                    c=colors_map, alpha=0.2, s=5)
    axes[1].axhline(PM25_THRESHOLD, color="red", linestyle="--", linewidth=1.2)
    axes[1].set_xlabel("Today's PM2.5 (µg/m³)")
    axes[1].set_ylabel("Tomorrow's PM2.5 (µg/m³)")
    axes[1].set_title("Today vs tomorrow PM2.5\n(red dots should be above the dashed line)")

    plt.suptitle("Target correctness visual check", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    # Quantitative check: among rows where target=1, what % has tomorrow's PM2.5 > threshold?
    pct_correct = (
        sample_check.loc[sample_check["pm25_exceedance_t1"] == 1, "pm25_tomorrow"] > PM25_THRESHOLD
    ).mean() * 100
    record("Target=1 rows have tomorrow's PM2.5 > 25 µg/m³",
           pct_correct > 99,
           f"{pct_correct:.2f}% of target=1 rows have tomorrow's PM2.5 above threshold")

    df_sorted.drop(columns=["pm25_tomorrow"], inplace=True)

---
## 7. Feature Distribution Checks

Verify that lag and rolling features have the correct statistical relationship to their source columns.  
If distributions are wildly different, it suggests a computation error in `spark_join_features.py`.

In [ ]:
# ---------------------------------------------------------------------------
# 7.1 Lag feature correlations with source column
#
# lag1 should have the highest correlation with the source (r ≈ autocorr at lag 1).
# lag2 should be slightly lower, lag7 lower still.
# If lag1 has a HIGHER correlation than expected (r > 0.99), it likely contains
# the same-day value — a leakage bug.
# ---------------------------------------------------------------------------
print("\n--- 7.1 Lag feature correlations ---")

lag_corr_results = []
for source_col in ["pm25_mean", "boundary_layer_height"]:
    if source_col not in df.columns:
        continue
    for lag_n in LAG_DAYS:
        lag_col = f"{source_col}_lag{lag_n}"
        if lag_col not in df.columns:
            continue
        valid = df[[source_col, lag_col]].dropna()
        if len(valid) < 1000:
            continue
        r = valid[source_col].corr(valid[lag_col])
        lag_corr_results.append({
            "source": source_col,
            "lag": lag_n,
            "correlation": round(r, 4),
            "suspicious": r > 0.99,
        })

corr_df = pd.DataFrame(lag_corr_results)
if len(corr_df) > 0:
    print(corr_df.to_string(index=False))
    suspicious = corr_df[corr_df["suspicious"]]
    record("Lag correlations are not suspiciously high (< 0.99)",
           len(suspicious) == 0,
           f"{len(suspicious)} lag features with r > 0.99 — possible same-day leakage"
           if len(suspicious) > 0 else "All correlations look reasonable")

In [ ]:
# ---------------------------------------------------------------------------
# 7.2 Rolling mean is always between min and max of source within the window
#
# A rolling mean can never exceed the maximum of the values it averages.
# If rolling7d > source_max (across the last 7 days), there is a computation error.
#
# We check this by verifying that rolling7d ≤ the 7-day rolling max.
# ---------------------------------------------------------------------------
print("\n--- 7.2 Rolling mean bounds ---")

if "pm25_mean_rolling7d" in df.columns and "pm25_mean" in df.columns:
    # Compute 7-day rolling max per station for comparison
    df_tmp = df.sort_values(["station_id", "date"]).copy()
    df_tmp["pm25_rolling7d_max"] = (
        df_tmp.groupby("station_id")["pm25_mean"]
        .transform(lambda x: x.shift(1).rolling(7, min_periods=1).max())
    )
    valid = df_tmp[["pm25_mean_rolling7d", "pm25_rolling7d_max"]].dropna()

    # Rolling mean must not exceed rolling max
    violations = (valid["pm25_mean_rolling7d"] > valid["pm25_rolling7d_max"] + 1e-6).sum()
    record("Rolling7d mean ≤ rolling7d max (no bound violations)",
           violations == 0,
           f"{violations:,} rows where rolling mean exceeds rolling max"
           if violations > 0 else f"OK ({len(valid):,} rows checked)")

---
## 8. Temporal Continuity

Verify that each station's time series has no unexpected gaps.  
A gap of more than one day in a station's time series will produce incorrect lag features for the rows immediately after the gap — the lag at t-1 will actually be t-2 or earlier.

`spark_clean_eea.py` does not fill gaps — it keeps raw data as-is.  
Gaps are expected for stations with low coverage, but large gaps in high-coverage stations are a data quality concern.

In [ ]:
# ---------------------------------------------------------------------------
# 8.1 Identify gaps in station time series
#
# For each station, compute the difference in days between consecutive dates.
# A gap > 1 day means at least one day of data is missing.
# ---------------------------------------------------------------------------
print("\n--- 8.1 Temporal continuity ---")

df_sorted = df[["station_id", "date", "country_code"]].sort_values(["station_id", "date"])
df_sorted["date_diff"] = (
    df_sorted.groupby("station_id")["date"]
    .diff()
    .dt.days
)

# Gaps > 1 day (excluding the first row of each station where diff is NaN)
gaps = df_sorted[df_sorted["date_diff"] > 1].copy()

pct_stations_with_gaps = gaps["station_id"].nunique() / df["station_id"].nunique() * 100

print(f"Stations with at least one gap: {gaps['station_id'].nunique():,} "
      f"({pct_stations_with_gaps:.1f}% of total)")
print(f"Total gap instances: {len(gaps):,}")

# For monitoring purposes — acceptable if < 20% of stations have gaps
record("< 20% of stations have temporal gaps",
       pct_stations_with_gaps < 20,
       f"{pct_stations_with_gaps:.1f}% of stations have at least one gap")

In [ ]:
# ---------------------------------------------------------------------------
# 8.2 Gap size distribution
#
# Short gaps (2–7 days) are expected — EEA stations sometimes miss a few days.
# Long gaps (> 30 days) are more serious — lag features spanning the gap
# will be misleading.
# ---------------------------------------------------------------------------
if len(gaps) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(gaps["date_diff"].clip(upper=60), bins=50,
            color="steelblue", edgecolor="white")
    ax.axvline(7,  color="orange", linestyle="--", linewidth=1.2, label="7 days")
    ax.axvline(30, color="red",    linestyle="--", linewidth=1.2, label="30 days")
    ax.set_xlabel("Gap length (days, clipped at 60)")
    ax.set_ylabel("Number of gap instances")
    ax.set_title("Distribution of temporal gaps in station time series")
    ax.legend()
    sns.despine()
    plt.tight_layout()
    plt.show()

    long_gaps = gaps[gaps["date_diff"] > 30]
    print(f"Gaps > 30 days: {len(long_gaps):,} instances in {long_gaps['station_id'].nunique()} stations")
    print("→ Consider flagging these stations or adding a 'days_since_last_reading' feature.")

    record("< 5% of gap instances are > 30 days",
           len(long_gaps) / max(len(gaps), 1) < 0.05,
           f"{len(long_gaps)/len(gaps)*100:.1f}% of gaps are > 30 days")

---
## 9. Baseline Model Experiment

The previous sections verified the structure and correctness of the feature store.  
This section verifies that the data is **learnable** — that a simple model can extract a meaningful signal from the features.

**This is not hyperparameter tuning.** We use a minimal LightGBM configuration with default settings.  
The goal is a sanity check: if the baseline model cannot beat a naive classifier, something is fundamentally wrong with the features or the target.

A naive classifier that always predicts 0 (no exceedance) achieves:  
- Accuracy ≈ (1 - positive_rate)%  
- Recall = 0%  

Our baseline must beat both. Recall > 50% confirms the features contain a useful signal.  
The full model in `train.py` targets Recall ≥ 90%.

In [ ]:
# ---------------------------------------------------------------------------
# 9.1 Prepare data for baseline experiment
#
# Temporal split:
#   train : 2019–2021
#   val   : 2022
#   test  : 2023 (held out — not used here)
#
# Feature set: a minimal set of same-day and lag-1 features only.
# Deliberately not including all features — we want to test the signal,
# not optimise performance.
# ---------------------------------------------------------------------------
try:
    import lightgbm as lgb
    from sklearn.metrics import recall_score, precision_score, roc_auc_score, classification_report
    LGBM_AVAILABLE = True
except ImportError:
    print("LightGBM not installed — skipping baseline experiment.")
    print("Install with: pip install lightgbm")
    LGBM_AVAILABLE = False

if LGBM_AVAILABLE:
    TARGET = "pm25_exceedance_t1"

    # Minimal feature set — deliberately conservative
    # The key question is: does lag1 alone give signal?
    baseline_features = [
        c for c in [
            "pm25_mean",
            "pm25_mean_lag1",
            "pm25_mean_lag2",
            "pm25_mean_lag7",
            "pm25_mean_rolling7d",
            "boundary_layer_height",
            "wind_speed",
            "temp_2m",
            "heating_season",
            "month",
        ] if c in df.columns
    ]

    print(f"Target          : {TARGET}")
    print(f"Baseline features ({len(baseline_features)}): {baseline_features}")

    # Temporal split
    train_mask = df["year"].isin([2019, 2020, 2021])
    val_mask   = df["year"] == 2022

    train_df = df[train_mask & df[TARGET].notna() & df[baseline_features].notna().all(axis=1)]
    val_df   = df[val_mask   & df[TARGET].notna() & df[baseline_features].notna().all(axis=1)]

    X_train = train_df[baseline_features]
    y_train = train_df[TARGET].astype(int)
    X_val   = val_df[baseline_features]
    y_val   = val_df[TARGET].astype(int)

    print(f"\nTrain: {len(X_train):,} rows  |  Val: {len(X_val):,} rows")
    print(f"Train positive rate: {y_train.mean()*100:.2f}%  |  Val positive rate: {y_val.mean()*100:.2f}%")

    # scale_pos_weight from EDA class imbalance
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    spw   = n_neg / n_pos
    print(f"scale_pos_weight  : {spw:.1f}")

In [ ]:
# ---------------------------------------------------------------------------
# 9.2 Train baseline LightGBM
#
# Minimal config — no tuning, no cross-validation.
# Early stopping on validation set prevents overfitting.
# ---------------------------------------------------------------------------
if LGBM_AVAILABLE:
    model = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        scale_pos_weight=spw,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(False)],
    )

    print(f"Best iteration: {model.best_iteration_}")

In [ ]:
# ---------------------------------------------------------------------------
# 9.3 Evaluate baseline model
#
# We tune the decision threshold to maximise recall while keeping
# precision above a minimum floor — mirroring the approach in train.py.
#
# Success criteria for the feature store validation:
#   - ROC AUC > 0.75  (better than random)
#   - Recall > 0.50   (better than naive always-predict-0)
# ---------------------------------------------------------------------------
if LGBM_AVAILABLE:
    # Get probability predictions
    y_prob = model.predict_proba(X_val)[:, 1]

    # Find threshold that maximises recall subject to precision > 0.2
    from sklearn.metrics import precision_recall_curve

    precisions, recalls, thresholds = precision_recall_curve(y_val, y_prob)
    # Filter to thresholds where precision > 0.2
    valid_mask = precisions[:-1] > 0.20
    if valid_mask.any():
        best_idx   = recalls[:-1][valid_mask].argmax()
        best_thresh = thresholds[valid_mask][best_idx]
    else:
        best_thresh = 0.5

    y_pred    = (y_prob >= best_thresh).astype(int)
    recall    = recall_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, zero_division=0)
    roc_auc   = roc_auc_score(y_val, y_prob)

    print("Baseline model results on validation set (2022):")
    print(f"  Decision threshold : {best_thresh:.3f}")
    print(f"  ROC AUC            : {roc_auc:.4f}")
    print(f"  Recall             : {recall*100:.2f}%")
    print(f"  Precision          : {precision*100:.2f}%")
    print("")
    print(classification_report(y_val, y_pred, target_names=["safe", "danger"]))

    record("Baseline ROC AUC > 0.75", roc_auc > 0.75,
           f"ROC AUC = {roc_auc:.4f}")
    record("Baseline Recall > 50%", recall > 0.50,
           f"Recall = {recall*100:.2f}% — feature store contains learnable signal"
           if recall > 0.50
           else f"Recall = {recall*100:.2f}% — investigate feature quality")

In [ ]:
# ---------------------------------------------------------------------------
# 9.4 Feature importance from baseline model
#
# Expected: lag features (especially lag1) should dominate.
# If a non-lag feature ranks #1 unexpectedly, investigate for leakage.
# ---------------------------------------------------------------------------
if LGBM_AVAILABLE:
    importance = pd.DataFrame({
        "feature":    baseline_features,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(
        importance["feature"][::-1],
        importance["importance"][::-1],
        color="steelblue", edgecolor="white",
    )
    ax.set_xlabel("Feature importance (LightGBM gain)")
    ax.set_title("Baseline model — feature importance\n"
                 "(pm25_mean_lag1 expected to dominate)")
    sns.despine()
    plt.tight_layout()
    plt.show()

    top_feature = importance.iloc[0]["feature"]
    record("Top feature is a lag or rolling pm25 feature",
           "pm25" in top_feature,
           f"Top feature: {top_feature}" +
           (" — unexpected, investigate for leakage" if "pm25" not in top_feature else ""))

---
## 10. Validation Summary

All checks recorded in the `VALIDATION_RESULTS` list are printed here as a pass/fail report.  
**The feature store is safe to use for model training only if all critical checks pass.**

In [ ]:
print("=" * 70)
print("FEATURE STORE VALIDATION REPORT")
print("=" * 70)

results_df = pd.DataFrame(VALIDATION_RESULTS)

n_pass = (results_df["status"] == "PASS").sum()
n_fail = (results_df["status"] == "FAIL").sum()
n_total = len(results_df)

for _, row in results_df.iterrows():
    icon = "✓" if row["status"] == "PASS" else "✗"
    detail = f" — {row['detail']}" if row["detail"] else ""
    print(f"  {icon} [{row['status']}] {row['check']}{detail}")

print("=" * 70)
print(f"Result: {n_pass}/{n_total} checks passed, {n_fail} failed")

# Critical checks — these must ALL pass before training
critical = [
    "No nulls in primary key 'station_id'",
    "No nulls in primary key 'date'",
    "No nulls in primary key 'country_code'",
    "No nulls in target 'pm25_exceedance_t1'",
    "No nulls in target 'pm25_exceedance_t2'",
    "No duplicate (station_id, date) rows",
    "Lag features look strictly backward (t-1)",
    "Target pm25_exceedance_t1 uses t+1 (not t)",
    "Target=1 rows have tomorrow's PM2.5 > 25 µg/m³",
    "Baseline ROC AUC > 0.75",
    "Baseline Recall > 50%",
]

critical_results = results_df[results_df["check"].isin(critical)]
critical_failures = critical_results[critical_results["status"] == "FAIL"]

print("")
if len(critical_failures) == 0:
    print("✓ All critical checks passed.")
    print("  → Feature store is SAFE to use for model training in train.py")
    print("  → Proceed to: src/model/train.py")
else:
    print("✗ CRITICAL FAILURES — do NOT train on this feature store.")
    for _, row in critical_failures.iterrows():
        print(f"  ✗ {row['check']} — {row['detail']}")
    print("  → Fix the root cause in spark_join_features.py and re-run the Spark job.")

print("=" * 70)